In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from upxo.pxtal.geometrification import GrainManifold2D
import upxo.gsdataops.grid_ops as gridOps

In [ ]:
lfi = gridOps.generate_test_2D_LFI_1(plotlfi=True)

In [ ]:
def get_boundary_seeds_v6(lfi, target_spacing=0.5, bulk_spacing=10.0, jitter_factor=1.0):
    """
    Seeding with Rigid Guard Rails to eliminate RVE boundary irregularities.
    Ensures straight edges at the domain limits [0, 200].
    """
    from scipy.ndimage import distance_transform_edt
    import numpy as np
    
    M, N = lfi.shape
    
    # 1. High-Resolution Interface Detection
    gy, gx = np.gradient(lfi)
    boundary_mask = (gx != 0) | (gy != 0)
    
    # 2. Equidistant Boundary Sampling
    coords_boundary = np.argwhere(boundary_mask).astype(float)
    stride = max(1, int(target_spacing / 1.0))
    seeds_boundary = coords_boundary[::stride]

    # 3. Jittered Bulk Sampling (Internal Only)
    dist_to_boundary = distance_transform_edt(~boundary_mask)
    # Define an internal buffer to prevent bulk seeds from pushing against RVE edges
    Y, X = np.indices((M, N))
    margin = 0.0
    internal_mask = (X > margin) & (X < N-margin) & (Y > margin) & (Y < M-margin)
    
    bulk_mask = (dist_to_boundary > target_spacing * 3) & internal_mask
    coords_bulk = np.argwhere(bulk_mask).astype(float)
    
    seeds_bulk = []
    if len(coords_bulk) > 0:
        bulk_stride = max(1, int(bulk_spacing))
        seeds_bulk = coords_bulk[::bulk_stride]
        # Apply jitter to break inclined/aliased patterns
        jitter = (np.random.rand(*seeds_bulk.shape) - 0.5) * (bulk_stride * jitter_factor)
        seeds_bulk += jitter

    # 4. RIGID GUARD RAILS (The Boundary Fix)
    # Place seeds exactly 1 unit outside the boundary to create flat bisectors
    guard_spacing = target_spacing 
    pad = 1.0 
    
    rail_coords = []
    # Horizontal rails (Top and Bottom)
    steps_x = np.arange(-pad, N + pad, guard_spacing)
    for x in steps_x:
        rail_coords.append([-pad, x])      # Bottom rail
        rail_coords.append([M + pad, x])   # Top rail
        
    # Vertical rails (Left and Right)
    steps_y = np.arange(-pad, M + pad, guard_spacing)
    for y in steps_y:
        rail_coords.append([y, -pad])      # Left rail
        rail_coords.append([y, N + pad])   # Right rail
    
    seeds_guard = np.array(rail_coords)
    all_seeds = np.vstack([seeds_boundary, seeds_bulk, seeds_guard])

    # 5. Standardize to [x, y] Cartesian coordinates
    return all_seeds[:, [1, 0]]

seeds = get_boundary_seeds_v6(lfi, target_spacing=0.5, bulk_spacing=5.0, jitter_factor=1.0)

plt.figure(figsize=(10, 10), dpi=150)
# plt.imshow(lfi, extent=[0, N, 0, M], origin='lower', alpha=0.3)
plt.plot(seeds[:, 0], seeds[:, 1], 'ro', label='Seeds', markersize=2)

In [ ]:
# 3. Initialize and Run Geometrification
# This uses the classmethod factory which handles Step 4, 5, and 6 internally
manifold_structure = GrainManifold2D.by_tessellation(lfi, seeds)

In [ ]:
manifold_structure.cells

In [ ]:
# 1. Store the original (jagged) state for comparison
original_cells = {gid: geom for gid, geom in manifold_structure.cells.items()}

In [ ]:
manifold_structure.smooth_interfaces(iterations=10, lmbda=0.5, mu=-0.53)

In [ ]:
import matplotlib.pyplot as plt
from shapely.geometry import MultiPolygon

def plot_manifold_geom(ax, cells_dict):
    """Helper to handle MultiPolygon and GeometryCollection iteration for plotting."""
    from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
    
    def get_polygons(g):
        """Recursively extract only Polygon parts from any geometry type."""
        if isinstance(g, Polygon):
            return [g]
        elif isinstance(g, (MultiPolygon, GeometryCollection)):
            res = []
            for part in g.geoms:
                res.extend(get_polygons(part))
            return res
        return [] # Filter out Points and LineStrings

    for gid, geom in cells_dict.items():
        # Clean the geometry to ensure we only have polygons
        polys = get_polygons(geom)

        # Plot each valid polygonal part
        first_part = True
        for part in polys:
            x, y = part.exterior.xy
            # Maintain consistent coloring for all parts of the same Grain ID
            if first_part:
                line, = ax.plot(x, y, linewidth=1.5, label=f'Grain {gid}')
                color = line.get_color()
                first_part = False
            else:
                ax.plot(x, y, linewidth=1.5, color=color)
            
            ax.fill(x, y, alpha=0.2, color=color)



In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), sharex=True, sharey=True, dpi=200)
# 1. Plot Original (Jagged)
ax1.set_title("Original (Voxel-Based Jagged)")
plot_manifold_geom(ax1, original_cells)

# 2. Plot Smoothed (Taubin)
ax2.set_title("Smoothed (Taubin Non-Shrinking)")
plot_manifold_geom(ax2, manifold_structure.cells)

# Formatting
for ax in [ax1, ax2]:
    ax.set_aspect('equal')
    ax.set_xlim(0, 200)
    ax.set_ylim(0, 200)
    # Removing duplicate legends for clarity, showing only one if needed
    # ax.legend(loc='upper right', fontsize='x-small', ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
manifold_structure.cells

In [ ]:
cells = manifold_structure.trim_to_rve(bounds=(6, 6, 194, 194))

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 7), sharex=True, sharey=True, dpi=200)
# 1. Plot Original (Jagged)
ax1.set_title("Original (Voxel-Based Jagged)")
plot_manifold_geom(ax1, original_cells)

# 2. Plot Smoothed (Taubin)
ax2.set_title("Smoothed (Taubin Non-Shrinking)")
plot_manifold_geom(ax2, manifold_structure.cells)

# 3. Plot Smoothed (Taubin) and cropped
ax3.set_title("Smoothed (Taubin Non-Shrinking) and Cropped")
plot_manifold_geom(ax3, cells)

# Formatting
for ax in [ax1, ax2, ax3]:
    ax.set_aspect('equal')
    ax.set_xlim(0, 200)
    ax.set_ylim(0, 200)
    # Removing duplicate legends for clarity, showing only one if needed
    # ax.legend(loc='upper right', fontsize='x-small', ncol=2)

plt.tight_layout()
plt.show()